# Control Node Setup — KVM@TACC

Provisions a persistent `m1.large` VM at KVM@TACC as the coachable-robots
control node. Run this notebook **once** from Chameleon JupyterHub or your laptop.
After provisioning, all day-to-day work happens on the control node.

## What gets created
- KVM@TACC `m1.large` instance (4 vCPU, 8 GB RAM, 80 GB disk)
- Blazar lease (up to 6 months — set `LEASE_MONTHS` below)
- Floating IP from `ext-net` (persistent)
- Security group allowing SSH inbound
- JupyterLab systemd service on port 8888
- coachable-robots repo cloned, `coachable` package installed
- SSH shortcut `pi` configured for the robot edge node

## Access after setup
```bash
ssh -L 8888:localhost:8888 cc@<kvm-floating-ip>
# open http://localhost:8888
```

## Prerequisites
- KVM@TACC application credential RC file (separate from CHI@TACC RC file)
  - Download from: https://kvm.tacc.chameleoncloud.org > Identity > Application Credentials
  - Save to: `ansible/app-cred-kvm-tacc-openrc.sh`
- Key pair registered at KVM@TACC site (separate from CHI@TACC key pairs)


## Resume — Run This First If Returning to an Existing Server

If the server is already provisioned, run this cell to restore all variables,
then skip to whichever cell you need (Ansible, verify, teardown, etc.).

In [3]:
import os, subprocess, time, socket, pathlib

# === Known state — update if server changes ===
SERVER_NAME  = 'coachable-robots-control'
LEASE_NAME   = 'coachable-robots-control'
SG_NAME      = 'coachable-robots-control-sg'
floating_ip  = '129.114.25.183'
ANSIBLE_DIR  = pathlib.Path('/home/ra/Projects/coachable-robots/ansible')
PRIVATE_KEY  = pathlib.Path.home() / '.ssh/id_rsa'

print(f'Floating IP: {floating_ip}')
print(f'SSH:         ssh cc@{floating_ip}')
print(f'Tunnel:      ssh -L 8888:localhost:8888 cc@{floating_ip}')
print('Variables set. Skip to the cell you need.')


Floating IP: 129.114.25.183
SSH:         ssh cc@129.114.25.183
Tunnel:      ssh -L 8888:localhost:8888 cc@129.114.25.183
Variables set. Skip to the cell you need.


## 0. Load KVM@TACC Credentials

KVM@TACC uses **separate** application credentials from CHI@TACC.

In [ ]:
# ✓ DONE — credentials loaded
# (cell commented out to prevent accidental re-run)

# import os, subprocess

# RC_FILE = os.path.expanduser("/home/ra/Projects/coachable-robots/ansible/app-cred-kvm-tacc-openrc.sh")

# if not os.path.exists(RC_FILE):
#     raise FileNotFoundError(
#         f"KVM@TACC RC file not found: {RC_FILE}\n"
#         "Download it from: https://kvm.tacc.chameleoncloud.org\n"
#         "  > Identity > Application Credentials > Create > Download openrc"
#     )

# result = subprocess.run(
#     ["bash", "-c", f"source {RC_FILE} && env"],
#     capture_output=True, text=True
# )
# for line in result.stdout.splitlines():
#     if line.startswith("OS_"):
#         key, _, val = line.partition("=")
#         os.environ[key] = val


# print("Credentials loaded:")
# for k, v in os.environ.items():
#     if k.startswith("OS_") and "SECRET" not in k and "PASSWORD" not in k:
#         print(f"  {k} = {v}")


## 1. Configure

In [ ]:
# ✓ DONE — chi configured
# (cell commented out to prevent accidental re-run)

# import chi
# from chi import lease, server
# from chi.lease import Lease
# from datetime import timedelta

# Reset any cached chi session before setting config
# import chi.context
# chi.context._session = None

# chi.use_site("KVM@TACC")
# project_name not needed — application credentials already encode project scope

# === CONFIGURE ===
# LEASE_NAME    = "coachable-robots-control"
# SERVER_NAME   = "coachable-robots-control"
# KEY_NAME      = "rick_rutgers_kvm"     # key pair registered at KVM@TACC
# IMAGE_NAME    = "CC-Ubuntu22.04"
# FLAVOR_NAME   = "m1.large"             # 4 vCPU, 8 GB RAM, 80 GB disk
# NETWORK_NAME  = "sharednet1"           # shared KVM@TACC tenant network
# FIP_POOL      = "ext-net"              # floating IP pool
# LEASE_MONTHS  = 6                      # up to 6 months for standard KVM flavors

# print(f"Site:        KVM@TACC")
# print(f"Project:     {chi.get('project_name')}")
# print(f"Lease:       {LEASE_NAME}  ({LEASE_MONTHS} months)")
# print(f"Server:      {SERVER_NAME}  ({FLAVOR_NAME})")
# print(f"Image:       {IMAGE_NAME}")
# print(f"Key:         {KEY_NAME}")


## 2. Create or Reuse Lease

KVM@TACC requires a Blazar lease even for VMs. Standard flavors support leases up to 6 months.

> **Note:** The `add_flavor_reservation()` API call may fail with an internal server error
> depending on KVM@TACC Blazar availability. If it does:
>
> 1. Check `kvm.tacc.chameleoncloud.org` > Reservations > Leases — delete any partial lease named `coachable-robots-control`
> 2. **Alternative: create the server manually via the dashboard:**
>    - Go to `kvm.tacc.chameleoncloud.org` > Compute > Instances > Launch Instance
>    - Flavor: `m1.large`, Image: `CC-Ubuntu22.04`, Key pair: `rick_rutgers_kvm`
>    - Network: `sharednet1`
>    - Security group: create one allowing TCP 22 inbound (or use the cell below to create it via API)
>    - After launch, assign a floating IP from `ext-net`
> 3. Set `floating_ip` manually and skip to cell 14 (Wait for SSH):
>    ```python
>    floating_ip = "YOUR.FLOATING.IP.HERE"
>    kvm_server = server.get_server(server.get_server_id(SERVER_NAME))
>    ```


In [ ]:
# ✓ DONE — lease created via dashboard
# (cell commented out to prevent accidental re-run)

# my_lease = None

# ── Check for existing lease ──
# print("Checking for existing leases...")
# all_leases = lease.list_leases()
# active_leases = [l for l in all_leases if l.status in ("ACTIVE", "PENDING")]

# for l in active_leases:
#     marker = " <<<" if l.name == LEASE_NAME else ""
#     print(f"  [{l.status}] {l.name}  ends: {l.end_date}{marker}")
#     if l.name == LEASE_NAME:
#         my_lease = l
#         print(f"Reusing existing lease '{LEASE_NAME}'.")

# if my_lease is None:
#     print(f"\nCreating new lease '{LEASE_NAME}' ({LEASE_MONTHS} months)...")
#     my_lease = Lease(
#         name=LEASE_NAME,
#         duration=timedelta(days=30 * LEASE_MONTHS),
#     )
#     # KVM flavor reservation — resource_type=flavor:instance
#     my_lease.add_flavor_reservation(name=FLAVOR_NAME, amount=1)
#     my_lease.submit(wait_for_active=True, idempotent=True)
#     print(f"Lease ACTIVE. Ends: {my_lease.end_date}")

# print(f"\nLease id: {my_lease.id}")


In [ ]:
# MANUAL OVERRIDE — server was created via dashboard
# Run this cell to set floating_ip and kvm_server before continuing

floating_ip = "129.114.25.183"
kvm_server = server.get_server(server.get_server_id(SERVER_NAME))
my_lease = None  # no lease object — created via dashboard
print(f'Control node: {floating_ip}')
print(f'Server id:    {kvm_server.id}')


## 3. Create Security Group

All KVM@TACC inbound traffic is blocked by default — SSH must be explicitly opened.

In [ ]:
# ✓ DONE — security group — check if created; may need manual creation too
# (cell commented out to prevent accidental re-run)

# from chi import clients

# neutron = clients.neutron()

# SG_NAME = "coachable-robots-control-sg"

# Check if security group already exists
# existing_sgs = neutron.list_security_groups(name=SG_NAME)['security_groups']

# if existing_sgs:
#     sg_id = existing_sgs[0]['id']
#     print(f"Security group '{SG_NAME}' already exists (id: {sg_id})")
# else:
#     sg = neutron.create_security_group({
#         "security_group": {
#             "name": SG_NAME,
#             "description": "coachable-robots control node — SSH + Jupyter tunnel"
#         }
#     })['security_group']
#     sg_id = sg['id']
#     print(f"Created security group '{SG_NAME}' (id: {sg_id})")

#     # Allow SSH inbound (port 22)
#     neutron.create_security_group_rule({
#         "security_group_rule": {
#             "security_group_id": sg_id,
#             "direction": "ingress",
#             "protocol": "tcp",
#             "port_range_min": 22,
#             "port_range_max": 22,
#             "remote_ip_prefix": "0.0.0.0/0",
#             "ethertype": "IPv4",
#         }
#     })
#     print("  Added rule: TCP 22 inbound (SSH)")

#     # Allow ICMP (ping) for diagnostics
#     neutron.create_security_group_rule({
#         "security_group_rule": {
#             "security_group_id": sg_id,
#             "direction": "ingress",
#             "protocol": "icmp",
#             "remote_ip_prefix": "0.0.0.0/0",
#             "ethertype": "IPv4",
#         }
#     })
#     print("  Added rule: ICMP inbound (ping)")

# print(f"\nSecurity group ready: {SG_NAME}")


## 4. Create or Reuse Server

In [ ]:
# ✓ DONE — server created via dashboard
# (cell commented out to prevent accidental re-run)

# import time

# kvm_server = None
# floating_ip = None

# ── Check for existing server ──
# try:
#     existing_id = server.get_server_id(SERVER_NAME)
#     existing = server.get_server(existing_id)
#     status = existing.status if hasattr(existing, 'status') else existing.get('status')
#     print(f"Found existing server '{SERVER_NAME}'  [{status}]")

#     if status == "ACTIVE":
#         kvm_server = existing
#         print("Reusing existing server.")
#     elif status == "SHUTOFF":
#         print("Server stopped — starting...")
#         server.start_server(existing_id)
#         server.wait_for_active(existing_id)
#         kvm_server = server.get_server(existing_id)
#         print("Server started.")
#     else:
#         print(f"Status '{status}' — check dashboard before proceeding.")

# except Exception:
#     print(f"No server named '{SERVER_NAME}' — creating...")

#     reservation_id = my_lease.flavor_reservations[0]['id']

#     kvm_server = server.create_server(
#         SERVER_NAME,
#         image_name=IMAGE_NAME,
#         flavor_name=FLAVOR_NAME,
#         key_name=KEY_NAME,
#         network_name=NETWORK_NAME,
#         security_groups=[SG_NAME],
#         scheduler_hints={"reservation": reservation_id},
#     )
#     print(f"Server created (id: {kvm_server.id}) — waiting for ACTIVE...")
#     server.wait_for_active(kvm_server.id)
#     print("Server ACTIVE.")


## 5. Assign Floating IP

In [ ]:
✓ DONE — floating IP 129.114.25.183 associated
(cell commented out to prevent accidental re-run)

from chi.network import list_floating_ips, associate_floating_ip, get_free_floating_ip

Check if server already has a floating IP
all_fips = list_floating_ips()
attached = [f for f in all_fips if f.get('instance_id') == kvm_server.id]

if attached:
    floating_ip = attached[0]['floating_ip_address']
    print(f"Server already has floating IP: {floating_ip}")
else:
    fip = get_free_floating_ip()
    if fip is None:
        from chi.network import allocate_floating_ip
        fip = allocate_floating_ip(FIP_POOL)
    floating_ip = fip['floating_ip_address']
    associate_floating_ip(kvm_server.id, floating_ip)
    print(f"Assigned floating IP: {floating_ip}")

print(f"\nSSH:    ssh cc@{floating_ip}")
print(f"Tunnel: ssh -L 8888:localhost:8888 cc@{floating_ip}")


## 6. Wait for SSH

In [4]:
import socket

print(f"Waiting for SSH on {floating_ip}:22 ...")
timeout, start = 300, time.perf_counter()

while True:
    try:
        with socket.create_connection((floating_ip, 22), timeout=10):
            print("SSH ready!")
            break
    except OSError:
        elapsed = time.perf_counter() - start
        if elapsed >= timeout:
            raise TimeoutError(f"SSH not ready after {timeout}s — check server console on dashboard")
        print(f"  {elapsed:.0f}s... retrying")
        time.sleep(10)


Waiting for SSH on 129.114.25.183:22 ...
  10s... retrying
  30s... retrying
  50s... retrying
  70s... retrying
  90s... retrying
  110s... retrying
  130s... retrying
  150s... retrying
  170s... retrying
  190s... retrying
  210s... retrying
  230s... retrying
  250s... retrying
  270s... retrying
  290s... retrying


TimeoutError: SSH not ready after 300s — check server console on dashboard

## 7. Verify Node

In [ ]:
from chi import ssh

with ssh.Remote(floating_ip) as conn:
    print("=== OS ===")
    conn.run("lsb_release -ds")
    print("\n=== Resources ===")
    conn.run("nproc && free -h | head -2 && df -h / | tail -1")


## 8. Run Ansible Playbook

Installs uv, JupyterLab (systemd), coachable package, HF CLI, and Pi SSH config.

> **Before running:** ensure `ansible/group_vars/all/vault.yml` contains `vault_hf_token` and `vault_pi_ssh_private_key`. Run `ansible-vault edit ansible/group_vars/all/vault.yml` to add them.

In [ ]:
import subprocess, pathlib

ANSIBLE_DIR         = pathlib.Path("/home/ra/Projects/coachable-robots/ansible")
PRIVATE_KEY         = pathlib.Path.home() / ".ssh/id_rsa"
VAULT_PASSWORD_FILE = ANSIBLE_DIR / ".vault_pass"

if not VAULT_PASSWORD_FILE.exists():
    raise FileNotFoundError(
        f"{VAULT_PASSWORD_FILE} not found.\n"
        "Create with: echo 'yourpassword' > ansible/.vault_pass && chmod 600 ansible/.vault_pass"
    )

inventory_content = f"""[control]
kvm ansible_host={floating_ip} ansible_user=cc ansible_ssh_private_key_file={PRIVATE_KEY}

[control:vars]
ansible_ssh_common_args=-o StrictHostKeyChecking=no
"""

inventory_path = ANSIBLE_DIR / "inventory_control.ini"
inventory_path.write_text(inventory_content)
print(f"Inventory: {inventory_path}")
print(inventory_content)


In [ ]:
result = subprocess.run(
    [
        "ansible-playbook",
        "-i", str(ANSIBLE_DIR / "inventory_control.ini"),
        "--vault-password-file", str(VAULT_PASSWORD_FILE),
        str(ANSIBLE_DIR / "playbooks/setup_control_node.yml"),
    ],
    cwd=str(ANSIBLE_DIR),
)

if result.returncode != 0:
    raise RuntimeError(f"Ansible failed (exit {result.returncode}) — scroll up for error details")
print("\nPlaybook complete.")


## 9. Verify and Print Access Info

In [ ]:
with ssh.Remote(floating_ip) as conn:
    print("=== JupyterLab service ===")
    conn.run("systemctl status jupyterlab --no-pager | head -15")
    print("\n=== coachable CLI ===")
    conn.run("coachable --help | head -6")
    print("\n=== HF CLI ===")
    conn.run("huggingface-cli whoami")
    print("\n=== Pi SSH ===")
    conn.run("ssh -o ConnectTimeout=5 pi 'echo Pi reachable' || echo 'Pi not reachable (check VPN/network)'")

print(f"""
====================================================
Control node ready.

From your laptop, open Jupyter:
  ssh -L 8888:localhost:8888 cc@{floating_ip}
  http://localhost:8888

Add to ansible/group_vars/all/vault.yml:
  vault_control_floating_ip: "{floating_ip}"

Lease ends: {my_lease.end_date}
Renew with: my_lease.extend(timedelta(days=30))
====================================================""")


## Save State to Vault

After provisioning, save the floating IP so it's available to other notebooks:

```bash
cd ansible
ansible-vault edit group_vars/all/vault.yml
```

Add:
```yaml
vault_control_floating_ip: "<paste floating_ip here>"
```

Also add to `group_vars/all/vars.yml`:
```yaml
control_floating_ip: "{{ vault_control_floating_ip }}"
```


## Teardown (when permanently decommissioning)

> Stop the server to pause billing. Only teardown fully if you're done with the project.

In [ ]:
action = input("Enter 'stop' to pause the server, 'delete' to remove everything, or press Enter to cancel: ").strip().lower()

if action == "stop":
    server.stop_server(kvm_server.id)
    print(f"Server '{SERVER_NAME}' stopped. Floating IP retained. Restart with cell 4.")

elif action == "delete":
    confirm = input("Type 'yes' to DELETE server and release floating IP: ").strip().lower()
    if confirm == "yes":
        from chi.network import release_floating_ip
        try:
            sid = server.get_server_id(SERVER_NAME)
            server.delete_server(sid)
            print(f"Server '{SERVER_NAME}' deleted.")
        except Exception as e:
            print(f"Server: {e}")
        try:
            release_floating_ip(floating_ip)
            print(f"Floating IP {floating_ip} released.")
        except Exception as e:
            print(f"FIP: {e}")
        try:
            lease.delete_lease(my_lease.id)
            print(f"Lease '{LEASE_NAME}' deleted.")
        except Exception as e:
            print(f"Lease: {e}")
    else:
        print("Cancelled.")
else:
    print("Cancelled.")
